In [22]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, SVR
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, \
RocCurveDisplay, roc_auc_score, r2_score, mean_absolute_error, f1_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import log_loss
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, BaggingClassifier, BaggingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import ridge_regression, ElasticNet
from sklearn.linear_model import Ridge


In [23]:
import warnings
warnings.filterwarnings('ignore')

importing df directly from uci website into pandas df

In [24]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
taiwanese_bankruptcy_prediction = fetch_ucirepo(id=572) 
  
# data (as pandas dataframes) 
X = taiwanese_bankruptcy_prediction.data.features 
y = taiwanese_bankruptcy_prediction.data.targets 
  
# metadata 
print(taiwanese_bankruptcy_prediction.metadata) 
  
# variable information 
print(taiwanese_bankruptcy_prediction.variables) 


{'uci_id': 572, 'name': 'Taiwanese Bankruptcy Prediction', 'repository_url': 'https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction', 'data_url': 'https://archive.ics.uci.edu/static/public/572/data.csv', 'abstract': 'The data were collected from the Taiwan Economic Journal  for the years 1999 to 2009. Company bankruptcy was defined based on the business regulations of the Taiwan Stock Exchange.', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 6819, 'num_features': 95, 'feature_types': ['Integer'], 'demographics': [], 'target_col': ['Bankrupt?'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Fri Mar 15 2024', 'dataset_doi': '10.24432/C5004D', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': None, 'purpose': None, 'funded_by': None, 'instances_represent': None, 'recommended_data_splits': None, 'sensitive_data': Non

In [25]:
y.value_counts()

Bankrupt?
0            6599
1             220
Name: count, dtype: int64

In [26]:
y.value_counts(normalize=True)*100

Bankrupt?
0            96.77372
1             3.22628
Name: proportion, dtype: float64

In [27]:
X

,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,Continuous interest rate (after tax),...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,0.780985,...,0.716845,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469
1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,0.781506,...,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794
2,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,0.780284,...,0.774670,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474
3,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,0.781241,...,0.739555,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982
4,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,0.781550,...,0.795016,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6814,0.493687,0.539468,0.543230,0.604455,0.604462,0.998992,0.797409,0.809331,0.303510,0.781588,...,0.799927,0.000466,0.623620,0.604455,0.840359,0.279606,0.027064,0.566193,1,0.029890
6815,0.475162,0.538269,0.524172,0.598308,0.598308,0.998992,0.797414,0.809327,0.303520,0.781586,...,0.799748,0.001959,0.623931,0.598306,0.840306,0.278132,0.027009,0.566018,1,0.038284
6816,0.472725,0.533744,0.520638,0.610444,0.610213,0.998984,0.797401,0.809317,0.303512,0.781546,...,0.797778,0.002840,0.624156,0.610441,0.840138,0.275789,0.026791,0.565158,1,0.097649
6817,0.506264,0.559911,0.554045,0.607850,0.607850,0.999074,0.797500,0.809399,0.303498,0.781663,...,0.811808,0.002837,0.623957,0.607846,0.841084,0.277547,0.026822,0.565302,1,0.044009


In [28]:
y

,Bankrupt?
0,1
1,1
2,1
3,1
4,1
...,...
6814,0
6815,0
6816,0
6817,0


In [29]:
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=25, test_size=0.3)

In [30]:
features = [2,3,4,5]
n_est = [25, 50, 100, 150, 200]
scores = []
for f in tqdm(features):
    for n in n_est:
        rf = RandomForestRegressor(random_state = 25, max_features=f, n_estimators=n)
        rf.fit(x_train, y_train)
        y_pred = rf.predict(x_test)
        scores.append([f,n,mean_absolute_error(y_test, y_pred)])

df_scores = pd.DataFrame(data = scores, columns = ['No. of Features', 'No of Estimators', 'score'])
df_scores.sort_values('score', ascending=True)

100%|██████████| 4/4 [00:24<00:00,  6.17s/it]


,No. of Features,No of Estimators,score
6,3,50,0.049638
8,3,150,0.049925
7,3,100,0.050098
9,3,200,0.050142
11,4,50,0.050156
5,3,25,0.050166
10,4,25,0.050420
18,5,150,0.050538
13,4,150,0.050557
12,4,100,0.050577


In [31]:
features = [2,3,4,5]
n_est = [25, 50, 100, 150, 200]
scores = []
for f in tqdm(features):
    for n in n_est:
        rf = RandomForestClassifier(random_state = 25, max_features=f, n_estimators=n)
        rf.fit(x_train, y_train)
        y_pred = rf.predict(x_test)
        scores.append([f,n,f1_score(y_test, y_pred, pos_label=1)])

df_scores = pd.DataFrame(data = scores, columns = ['No. of Features', 'No of Estimators', 'score'])
df_scores.sort_values('score', ascending=False)

100%|██████████| 4/4 [00:27<00:00,  6.98s/it]


,No. of Features,No of Estimators,score
10,4,25,0.340909
11,4,50,0.285714
2,2,100,0.282051
6,3,50,0.275000
17,5,100,0.265060
19,5,200,0.258824
15,5,25,0.258065
18,5,150,0.255814
3,2,150,0.253165
0,2,25,0.250000


In [32]:
df_scores.sort_values('score', ascending=False)

,No. of Features,No of Estimators,score
10,4,25,0.340909
11,4,50,0.285714
2,2,100,0.282051
6,3,50,0.275000
17,5,100,0.265060
19,5,200,0.258824
15,5,25,0.258065
18,5,150,0.255814
3,2,150,0.253165
0,2,25,0.250000


Classification Report

In [34]:
best_model = RandomForestClassifier(random_state = 25, max_features=4, n_estimators=25)
best_model.fit(x_train, y_train)
y_pred = best_model.predict(x_test)
# scores.append([f,n,f1_score(y_test, y_pred, pos_label=1)])
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1983
           1       0.60      0.24      0.34        63

    accuracy                           0.97      2046
   macro avg       0.79      0.62      0.66      2046
weighted avg       0.96      0.97      0.97      2046



using wine dataset

In [35]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
wine = fetch_ucirepo(id=109) 
  
# data (as pandas dataframes) 
X = wine.data.features 
y = wine.data.targets 
  
# metadata 
print(wine.metadata) 
  
# variable information 
print(wine.variables) 


{'uci_id': 109, 'name': 'Wine', 'repository_url': 'https://archive.ics.uci.edu/dataset/109/wine', 'data_url': 'https://archive.ics.uci.edu/static/public/109/data.csv', 'abstract': 'Using chemical analysis to determine the origin of wines', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 178, 'num_features': 13, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1992, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C5PC7J', 'creators': ['Stefan Aeberhard', 'M. Forina'], 'intro_paper': {'ID': 246, 'type': 'NATIVE', 'title': 'Comparative analysis of statistical pattern recognition methods in high dimensional settings', 'authors': 'S. Aeberhard, D. Coomans, O. Vel', 'venue': 'Pattern Recognition', 'year': 1994, 'journal': None, 'DOI': '10.1016/0031-3203(94)90145-7', 'URL': 'https:

In [36]:
y.value_counts()

class
2        71
1        59
3        48
Name: count, dtype: int64